In [6]:
import pandas as pd
import numpy as np
from typing import Any
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler,OneHotEncoder,OrdinalEncoder
from sklearn.impute import SimpleImputer

In [7]:
df = pd.read_csv("../data/raw/student_mental_health.csv")
df.head()

,Student_ID,Age,Gender,Year_of_Study,Sleep_Hours,Study_Hours_Per_Week,GPA,Social_Media_Hours,Screen_Time_Hours,Physical_Activity,Financial_Stress,Family_Support,Attendance_Rate,Part_Time_Job,Diet_Quality,Previous_Mental_Health_History,Mental_Health_Risk
0,STU0001,22,Male,3,6.6,17.6,2.91,6.9,7.5,Moderate,8.5,4.0,78.0,No,Poor,No,High
1,STU0002,21,Male,4,5.6,22.4,2.92,7.6,11.5,Low,4.5,5.3,85.4,Yes,Average,No,Medium
2,STU0003,22,Male,5,7.1,28.1,2.68,6.5,8.9,Low,4.8,6.1,79.5,No,Average,No,Medium
3,STU0004,24,Male,1,4.5,14.8,2.23,5.5,7.6,High,3.7,8.0,73.9,No,Average,Yes,High
4,STU0005,20,Female,1,6.7,17.2,2.69,4.3,7.0,Moderate,8.0,7.4,99.8,Yes,NaN,No,Medium


In [8]:
df = df.drop(columns=["Student_ID","Mental_Health_Risk"])

In [9]:

def data_split(df: pd.DataFrame): 

    x = df.drop(columns="Mental_Health_Risk")
    y = df["Mental_Health_Risk"]

    x_train , x_test , y_train , y_test = train_test_split(
            x,
            y,
            test_size= 0.2,
            random_state= 42
        )

    return x_train , x_test , y_train , y_test

x_train , x_test , y_train , y_test = data_split(df)

KeyError: "['Mental_Health_Risk'] not found in axis"

In [10]:
def preprocessing(
    numerical_features: list[str],
    ordinal_features: list[str],
    nominal_features: list[str],
    ordinal_categories: list[list[str]] 
):

    tranformers = []

    if numerical_features:
        numerical_pipeline = (
            [
                ("imputer",SimpleImputer(strategy="median")),
                ("scaler" , StandardScaler())
            ]
        )

        tranformers.append(("numerical",numerical_pipeline,numerical_features))

    if ordinal_features:
        ordinal_pipeline = (
            [
                ("imputer",SimpleImputer(strategy="most_frequent")),
                ("encoder",OrdinalEncoder(categories=ordinal_categories,handle_unknown="use_encoded_value",unknown_value=1))
            ]
        )

        tranformers.append(("ordinal",ordinal_pipeline,ordinal_features))

    if nominal_features:
        nominal_pipeline = (
            [
                ("imputer",SimpleImputer(strategy="most_frequent")),
                ("encoder",OneHotEncoder(handle_unknown="ignore"))
            ]
        )

        tranformers.append(("nominal",nominal_pipeline,nominal_features))

    preprocessor = ColumnTransformer(
        transformers=tranformers,
        remainder="drop"
    )

    return preprocessor

In [11]:
numerical_features = df.select_dtypes(include=np.number).columns
ordinal_features = ["Physical_Activity" , "Diet_Quality"]
nominal_features =  ["Gender" , "Part_Time_Job" , "Previous_Mental_Health_History"]
ordinal_categories = [
    ['Low', 'Moderate', 'High'], 
    ['Poor', 'Average', 'Good']
]

preprocessor = preprocessing(nominal_features,ordinal_features,nominal_features,ordinal_categories)